# PoisonSpot × Narcissus — Facial Emotion Recognition (Kaggle)
**ResNet18 | CIS 582 | seed 42**

Full pipeline: trigger gen → poisoned training → batch-level provenance → sample-level provenance → scoring → retrain.

**Only one dataset required:** `sujaykapadnis/emotion-recognition-dataset` (attach before running).
All patched source files are embedded in this notebook.

The CONFIG cell below targets the **quick smoke-test** (5 epochs). Edit the full-run values before the final experiment run.

In [ ]:
# ════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════

import os as _os

# Auto-detect the emotion dataset class-folder root.
# Handles both flat  (dataset/angry/, dataset/happy/, …)
# and split layouts  (dataset/train/angry/, dataset/train/happy/, …).
_base = '/kaggle/input/emotion-recognition-dataset'
if not _os.path.exists(_base):
    # Fallback: scan /kaggle/input for any emotion-like dataset
    for _d in sorted(_os.listdir('/kaggle/input')):
        _candidate = _os.path.join('/kaggle/input', _d)
        if _os.path.isdir(_candidate):
            _base = _candidate
            break

def _find_emotion_cfg(base):
    """Walk base and return the first dir with >=4 subdirs containing images."""
    IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
    for root, dirs, _ in _os.walk(base):
        dirs.sort()
        if len(dirs) < 2:
            continue
        sub = _os.path.join(root, dirs[0])
        try:
            contents = _os.listdir(sub)
        except PermissionError:
            continue
        if any(_os.path.splitext(f)[1].lower() in IMG_EXTS for f in contents):
            return root
    return base

EMOTION_DIR  = _find_emotion_cfg(_base)
print('EMOTION_DIR auto-detected:', EMOTION_DIR)

REPO_DIR     = '/kaggle/working/PoisonSpot'
OUT_DIR      = '/kaggle/working/outputs'
SAVED_MODELS = f'{OUT_DIR}/saved_models'
PROV_PATH    = f'{OUT_DIR}/provenance'
RESULTS_PATH = f'{OUT_DIR}/results'
CFG_PATH     = f'{REPO_DIR}/configs/run_config_emotion.yaml'

# ── Attack / detection ───────────────────────────────────
# Class indices depend on alphabetical sort of emotion class folders:
# 0=angry 1=disgust 2=fear 3=happy 4=neutral 5=sad 6=surprise (typical)
TARGET_CLASS     = 3    # 'happy' by default (well-populated class)
PR_TGT           = 10  # % of target-class train images to poison
PR_SUS           = 50  # suspected-sample budget (%)
EPS              = 16  # L-inf trigger budget (0-255)
IMG_SIZE         = 112 # resize to this before feeding ResNet18

# ── Quick smoke test (5 epochs) ────────────────────────
# Swap for full-run values once smoke test passes.
SURROGATE_EPOCHS = 5
GEN_STEPS        = 200
EPOCHS           = 30
BATCH_SIZE       = 64
LR               = 0.01
GLOBAL_SEED      = 42
EP_BL_BASE       = 50
EP_BL            = 5
EP_SL_BASE       = 25
EP_SL            = 5
BS_BL            = 64
BS_SL            = 16
GROUPS           = 15
CV_MODEL         = 'RandomForest'
FORCE            = False


In [ ]:
import subprocess, sys, os

try:
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True)
    print('GPU:', r.stdout.strip() or 'NOT FOUND')
except FileNotFoundError:
    print('nvidia-smi not on PATH — trying torch')

import torch
print('CUDA:', torch.cuda.is_available(), '| PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected')

# captum is required by scoring.py
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyyaml', 'tqdm', 'captum', '--no-deps'], check=True)
print('deps ok')


In [ ]:
import os, subprocess

# Upstream is pinned. `--depth 1` on the default branch would run whatever
# HEAD happened to be that day, and the patch below is written against one
# specific commit -- if upstream moved, it would apply to code it was never
# written for, or fail, and neither is visible from the results.
UPSTREAM_SHA = 'fe5590228124f98d4abb2032aefc1f78075a1b87'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/Philenku/PoisonSpot', REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--quiet', UPSTREAM_SHA], check=True)
head = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                      capture_output=True, text=True).stdout.strip()
assert head == UPSTREAM_SHA, f'upstream at {head}, expected {UPSTREAM_SHA}'
print('upstream pinned at', head[:9])


In [ ]:
import os, subprocess, urllib.request

# The extension, as a diff against the pinned upstream commit.
# This replaces six cells that each wrote a base64 blob over an upstream
# file. Those worked, but they made the contribution unreadable -- 195 KB
# of encoded payload that could not be reviewed or diffed -- and they
# redistributed upstream code that carries no licence.
PATCH_URL = ('https://raw.githubusercontent.com/craft-b/PoisonSpot/'
             'master/patches/fer-narcissus.patch')
PATCH_PATH = '/kaggle/working/fer-narcissus.patch'

local = os.path.join(os.getcwd(), 'patches', 'fer-narcissus.patch')
if os.path.exists(local):
    PATCH_PATH = local
    print('using patch from the working tree')
else:
    urllib.request.urlretrieve(PATCH_URL, PATCH_PATH)
    print('downloaded patch')

subprocess.run(['git', '-C', REPO_DIR, 'apply', '--check', PATCH_PATH], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'apply', PATCH_PATH], check=True)
print('patch applied cleanly')

changed = subprocess.run(['git', '-C', REPO_DIR, 'diff', '--stat'],
                         capture_output=True, text=True).stdout
print(changed)


In [ ]:
import subprocess
subprocess.run(
    ['find', REPO_DIR, '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
    capture_output=True)
print('pycache cleared')


In [ ]:
import os, yaml

for d in [SAVED_MODELS, PROV_PATH, RESULTS_PATH, os.path.dirname(CFG_PATH)]:
    os.makedirs(d, exist_ok=True)

cfg = {
    'exp':                  f'emotion_narcissus_eps{EPS}_tgt{PR_TGT}_seed{GLOBAL_SEED}',
    'attack':               'narcissus',
    'dataset':              'emotion',
    'model':                'ResNet18',
    'scenario':             'from_scratch',
    'dataset_dir':          EMOTION_DIR,
    'saved_models_path':    SAVED_MODELS + '/',
    'prov_path':            PROV_PATH + '/',
    'results_path':         RESULTS_PATH + '/',
    'clean_model_path':     '',
    'poisoned_model_path':  '',
    'retrained_model_path': '',
    'target_class':         TARGET_CLASS,
    'source_class':         TARGET_CLASS,
    'pr_tgt':               PR_TGT,
    'pr_sus':               PR_SUS,
    'eps':                  EPS,
    'img_size':             IMG_SIZE,
    'surrogate_epochs':     SURROGATE_EPOCHS,
    'gen_steps':            GEN_STEPS,
    'epochs':               EPOCHS,
    'bs':                   BATCH_SIZE,
    'lr':                   LR,
    'opt':                  'sgd',
    'global_seed':          GLOBAL_SEED,
    'gpu_id':               0,
    'clean_training':       False,
    'poisoned_training':    True,
    'batch_level':          True,
    'sample_level':         True,
    'score_samples':        True,
    'retrain':              True,
    'ep_bl':                EP_BL,
    'ep_bl_base':           EP_BL_BASE,
    'ep_sl':                EP_SL,
    'ep_sl_base':           EP_SL_BASE,
    'bs_bl':                BS_BL,
    'bs_sl':                BS_SL,
    'k_1':                  1,
    'k_2':                  0.0001,
    'groups':               GROUPS,
    'cv_model':             CV_MODEL,
    'threshold_type':       'Kmeans',
    'custom_threshold':     0.5,
    'vis':                  False,
    'get_result':           False,
    'force':                FORCE,
    'random':               False,
    'sample_from_test':     False,
}

with open(CFG_PATH, 'w') as fh:
    yaml.dump(cfg, fh, default_flow_style=False, sort_keys=False)
print(f'Config written: {CFG_PATH}')
print(f'epochs={EPOCHS}  ep_bl_base={EP_BL_BASE}  ep_sl_base={EP_SL_BASE}')


In [ ]:
import sys, os

# Verify dataset was found and has at least 2 class dirs
classes = [d for d in os.listdir(EMOTION_DIR)
           if os.path.isdir(os.path.join(EMOTION_DIR, d))]
print(f'Emotion dataset: {len(classes)} classes found')
print('  ->', sorted(classes))
assert len(classes) >= 2, f'Expected >=2 emotion classes, got {len(classes)}'
assert TARGET_CLASS < len(classes), \
    f'TARGET_CLASS={TARGET_CLASS} but only {len(classes)} classes present'

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR

from src import (
    get_narcissus_emotion_poisoned_data,
    get_loaders_from_dataset, train, evaluate_model,
    capture_first_level_multi_epoch_batch_sample_weight_updates,
    capture_sample_level_weight_updates_idv,
    train_prov_data_custom, score_poisoned_samples, get_diff,
)
print('src imports OK')
print('Preflight passed — ready to run')


## Run pipeline

**Quick smoke test** (~15–30 min on T4): 5 epochs, confirms end-to-end on Kaggle.

Once confirmed, edit Cell 2 for the **full experiment run**:
```python
SURROGATE_EPOCHS = 5
GEN_STEPS        = 200
EPOCHS           = 30
BATCH_SIZE       = 64
EP_BL_BASE       = 25
EP_BL            = 5
EP_SL_BASE       = 25
EP_SL            = 5
BS_BL            = 64
BS_SL            = 16
GROUPS           = 10
FORCE            = False
```

**Target class note:** classes are sorted alphabetically by folder name. Default `TARGET_CLASS=3` targets `happy` (index 3 in angry/disgust/fear/happy/…). Verify with the preflight cell output and adjust if your dataset uses a different ordering.

In [ ]:
import subprocess, os, threading, time

env = os.environ.copy()
env["PYTHONPATH"]       = REPO_DIR
env["PYTHONIOENCODING"] = "utf-8"

proc = subprocess.Popen(
    ["python", "-u", "main.py", "-c", CFG_PATH],
    cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

def heartbeat():
    while proc.poll() is None:
        time.sleep(30)
        if proc.poll() is None:
            print(".", end="", flush=True)

t = threading.Thread(target=heartbeat, daemon=True)
t.start()

for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print("
Exit code:", proc.returncode)
if proc.returncode != 0:
    raise RuntimeError("Pipeline failed -- check output above")


In [ ]:
import glob, os

print('=== Saved models ===')
for f in sorted(glob.glob(os.path.join(SAVED_MODELS, '*.pkl'))):
    print(' ', os.path.basename(f))

print('\n=== Results ===')
for f in sorted(glob.glob(os.path.join(RESULTS_PATH, '**'), recursive=True)):
    if os.path.isfile(f):
        print(' ', f.replace(RESULTS_PATH + '/', ''))

print('\n=== Provenance pkls ===')
for f in sorted(glob.glob(os.path.join(PROV_PATH, '*.pkl'))):
    print(' ', os.path.basename(f))
